# ML Surrogates for Chemical Processes with Gurobi-ML

This notebook illustrates the use of TensorFlow Keras and Gurobi-ML to produce an ML surrogate based on data from a chemical process flowsheet.

**Note:** This notebook is adapted from the [OMLT Auto-thermal Reformer example](https://github.com/cog-imperial/OMLT/blob/main/docs/notebooks/Thermal/auto-thermal-reformer.ipynb). The data is sourced from the OMLT repository.

There are several reasons to build surrogate models for complex processes, even when higher fidelity models already exist (e.g., reduce model size, improve convergence reliability, replace models with externally compiled code and make them fully-equation oriented).

## 1. Setup and Configuration

In [1]:
import os
import urllib.request
import pandas as pd
import gurobipy as gp
import gurobipy_pandas as gppd
from tensorflow import keras
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from gurobi_ml import add_predictor_constr

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# --- Configuration ---
DATA_URL = "https://raw.githubusercontent.com/cog-imperial/OMLT/main/docs/notebooks/data/reformer.csv"
DATA_FILE = "reformer.csv"

INPUT_COLS = ["Bypass Fraction", "NG Steam Ratio"]
OUTPUT_COLS = [
    "Steam Flow",
    "Reformer Duty",
    "AR",
    "C2H6",
    "C3H8",
    "C4H10",
    "CH4",
    "CO",
    "CO2",
    "H2",
    "H2O",
    "N2",
]

# Neural Network Parameters
N_LAYERS = 4
N_NEURONS = 20
LEARNING_RATE = 0.01
EPOCHS = 20

I0000 00:00:1778268305.571431   54164 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 2. Data Loading and Preprocessing

In [2]:
def load_reformer_data(url, filename):
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filename)
    return pd.read_csv(filename, usecols=INPUT_COLS + OUTPUT_COLS)


df = load_reformer_data(DATA_URL, DATA_FILE)
dfin, dfout = df[INPUT_COLS], df[OUTPUT_COLS]

# Scale data using Scikit-Learn
scaler_in = StandardScaler().fit(dfin)
scaler_out = StandardScaler().fit(dfout)

X_train = scaler_in.transform(dfin)
y_train = scaler_out.transform(dfout)

# Capture physical bounds for optimization
input_bounds = pd.DataFrame({"lb": dfin.min(), "ub": dfin.max()})

df.head()

,Bypass Fraction,NG Steam Ratio,Steam Flow,Reformer Duty,AR,C2H6,C3H8,C4H10,CH4,CO,CO2,H2,H2O,N2
0,0.8,0.800000,0.193898,9806.732716,0.002662,0.012120,0.002651,0.001515,0.369276,0.073971,0.032251,0.208494,0.070771,0.226288
1,0.8,0.810526,0.196449,9846.047501,0.002660,0.012107,0.002648,0.001513,0.368883,0.073684,0.032432,0.208507,0.071514,0.226050
2,0.8,0.821053,0.199000,9885.419259,0.002657,0.012094,0.002646,0.001512,0.368491,0.073398,0.032612,0.208519,0.072258,0.225813
3,0.8,0.831579,0.201552,9924.849127,0.002654,0.012082,0.002643,0.001510,0.368100,0.073114,0.032791,0.208529,0.073000,0.225577
4,0.8,0.842105,0.204103,9964.338177,0.002651,0.012069,0.002640,0.001509,0.367710,0.072832,0.032968,0.208537,0.073743,0.225341


## 3. Surrogate Model Training

In [3]:
# Build parameterized Keras model
nn = Sequential(
    [
        keras.Input(shape=(len(INPUT_COLS),)),
        *[Dense(N_NEURONS, activation="sigmoid") for _ in range(N_LAYERS)],
        Dense(len(OUTPUT_COLS)),
    ],
    name=f"reformer_sigmoid_{N_LAYERS}_{N_NEURONS}",
)

nn.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss="mse")

# Show architecture
nn.summary()

E0000 00:00:1778268307.775157   54164 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_COMPAT_NOT_SUPPORTED_ON_DEVICE: forward compatibility was attempted on non supported HW
I0000 00:00:1778268307.775182   54164 cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
I0000 00:00:1778268307.775184   54164 cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: pop-os
I0000 00:00:1778268307.775188   54164 cuda_diagnostics.cc:183] hostname: pop-os
I0000 00:00:1778268307.775254   54164 cuda_diagnostics.cc:190] libcuda reported version is: 580.159.3
I0000 00:00:1778268307.775271   54164 cuda_diagnostics.cc:194] kernel reported version is: 580.126.18
E0000 00:00:1778268307.775274   54164 cuda_diagnostics.cc:287] kernel version 580.126.18 does not match DSO version 580.159.3 -- cannot find working devices 

Model: "reformer_sigmoid_4_20"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 20)             │            60 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │           420 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 20)             │           420 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 20)             │           420 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 12)             │           252 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,572 (6.14 KB)

 Trainable params: 1,572 (6.14 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
print(f"\nTraining {nn.name}...")
history = nn.fit(X_train, y_train, epochs=EPOCHS, validation_split=0.2, verbose=2)

print(f"\nFinal Training Loss:   {history.history['loss'][-1]:.6f}")
print(f"Final Validation Loss: {history.history['val_loss'][-1]:.6f}")


Training reformer_sigmoid_4_20...
Epoch 1/20
70/70 - 1s - 17ms/step - loss: 0.8152 - val_loss: 0.7087
Epoch 2/20
70/70 - 0s - 3ms/step - loss: 0.1279 - val_loss: 0.3061
Epoch 3/20
70/70 - 0s - 2ms/step - loss: 0.0531 - val_loss: 0.1975
Epoch 4/20
70/70 - 0s - 2ms/step - loss: 0.0162 - val_loss: 0.1102
Epoch 5/20
70/70 - 0s - 2ms/step - loss: 0.0088 - val_loss: 0.0792
Epoch 6/20
70/70 - 0s - 2ms/step - loss: 0.0041 - val_loss: 0.0487
Epoch 7/20
70/70 - 0s - 2ms/step - loss: 0.0024 - val_loss: 0.0362
Epoch 8/20
70/70 - 0s - 2ms/step - loss: 0.0018 - val_loss: 0.0286
Epoch 9/20
70/70 - 0s - 2ms/step - loss: 0.0013 - val_loss: 0.0254
Epoch 10/20
70/70 - 0s - 2ms/step - loss: 0.0012 - val_loss: 0.0229
Epoch 11/20
70/70 - 0s - 2ms/step - loss: 8.8940e-04 - val_loss: 0.0194
Epoch 12/20
70/70 - 0s - 2ms/step - loss: 8.4424e-04 - val_loss: 0.0181
Epoch 13/20
70/70 - 0s - 2ms/step - loss: 6.8993e-04 - val_loss: 0.0171
Epoch 14/20
70/70 - 0s - 2ms/step - loss: 6.7805e-04 - val_loss: 0.0153
Epoch

## 4. Optimization Problem

We seek to maximize Hydrogen concentration (`H2`) while keeping Nitrogen (`N2`) below 0.34.

In [6]:
m = gp.Model("Reformer_Optimization")

# Variables in physical space (indexed by column names)
x = gppd.add_vars(m, input_bounds, lb="lb", ub="ub", name="x")
y = gppd.add_vars(m, dfout.columns, lb=-gp.GRB.INFINITY, name="y")

# Intermediate scaled variables
x_scaled = gppd.add_vars(m, x.index, lb=-gp.GRB.INFINITY, name="x_scaled")
y_scaled = gppd.add_vars(m, y.index, lb=-gp.GRB.INFINITY, name="y_scaled")

# Connect variables via Gurobi-ML pipeline
# Gurobi-ML supports gurobipy-pandas objects (Series/DataFrames)
add_predictor_constr(m, scaler_in, x, x_scaled)
add_predictor_constr(m, nn, x_scaled, y_scaled)
add_predictor_constr(m, scaler_out, y, y_scaled)

# Objective and constraints using column names
m.setObjective(y["H2"], gp.GRB.MAXIMIZE)
m.addConstr(y["N2"] <= 0.34)

m.Params.OptimalityTarget = 1

m.optimize()

Set parameter OptimalityTarget to value 1
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Pop!_OS 24.04 LTS")

CPU model: Intel(R) Core(TM) i5-4460  CPU @ 3.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 4 logical processors, using up to 4 threads

Non-default parameters:
OptimalityTarget  1

WLS license 2788429 - registered to Gurobi GmbH
Optimize a model with 107 rows, 188 columns and 1601 nonzeros (Max)
Model fingerprint: 0x70f53532
Model has 1 linear objective coefficients
Model has 80 general nonlinear constraints (80 nonlinear terms)
Variable types: 188 continuous, 0 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e-05, 8e+03]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e-01, 1e+00]
  RHS range        [5e-04, 2e+04]

Presolve removed 26 rows and 26 columns
Presolve time: 0.00s
Presolved: 81 rows, 162 columns, 1340 nonzeros
Presolved model has 80 nonlinear constraint(s)

Solving NLP to local optimality

Ordering time:

## 5. Results

In [11]:
if m.status in (gp.GRB.OPTIMAL, gp.GRB.LOCALLY_OPTIMAL):
    print("\nOptimal Operating Point:")
    print(x.gppd.X)

    print("\nPredicted Concentrations:")
    print(f"{'H2':>16}: {y['H2'].X:.4f} (Maximized)")
    print(f"{'N2':>16}: {y['N2'].X:.4f} (Constraint: <= 0.34)")


Optimal Operating Point:
Bypass Fraction    0.10000
NG Steam Ratio     1.14702
Name: x, dtype: float64

Predicted Concentrations:
              H2: 0.3323 (Maximized)
              N2: 0.3400 (Constraint: <= 0.34)
